In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
import polars as pl
from datetime import datetime, timedelta


In [ ]:
# Load scores and incidents df
dir = Path('/home/nikita/carGPT/inference_results')
model_name = 'model-uv3u2e5g:v3'
inference_datetime = '2024-05-31_12-27-48'

work_dir = dir / model_name / inference_datetime
assert work_dir.exists(), f'No such directory: {work_dir}'

df = pd.read_csv(work_dir / 'scores.csv')
df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')

df_inc = pd.read_csv(work_dir / 'incidents.csv')
df_inc['datetime']  = pd.to_datetime(df_inc.datetime).apply(lambda x: x.replace(tzinfo=None))


In [ ]:
# Extract only specific drive and action scores
score_type = 'score_l1'
action = 'gas_pedal'

drive_id = df_inc.drive_id.unique()[0]

df_drive = df[df.drive_id == drive_id]
df_action = df_drive[(df_drive[score_type].notna()) & (df_drive.name == action)].sort_values(by='datetime', ascending=True)
df_action

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(
    data = go.Scatter(
        x = df_action.datetime,
        y = df_action[score_type],
    )
)
for i, row in df_inc.query("drive_id == @drive_id").iterrows():
    fig.add_vline(x=row.datetime, line_dash="dash", line_color="red", line_width=1)



fig.show()

In [79]:

# Create the first DataFrame with 1000 rows
df1 = pl.DataFrame({
    "timestamp": [datetime(2024, 1, 1, 0, 0, 0) + timedelta(minutes=i) for i in range(1000)],
    "value1": range(1000)
})

# Create the second DataFrame with 10 rows
df2 = pl.DataFrame({
    "timestamp": [datetime(2024, 1, 1, 0, 0, 2) + timedelta(minutes=i*100) for i in range(10)],
    "inc": range(10)
})

# Perform an asof join to find the closest timestamp
df = df1.sort(by='timestamp').join_asof(df2.sort(by='timestamp'), on="timestamp", strategy="nearest", tolerance='5s')

# Handle the rest of the values setting to null where necessary
# Here, the asof join already sets unmatched values to null, so additional steps might not be required
print(df)


shape: (1_000, 3)
┌─────────────────────┬────────┬──────┐
│ timestamp           ┆ value1 ┆ inc  │
│ ---                 ┆ ---    ┆ ---  │
│ datetime[μs]        ┆ i64    ┆ i64  │
╞═════════════════════╪════════╪══════╡
│ 2024-01-01 00:00:00 ┆ 0      ┆ 0    │
│ 2024-01-01 00:01:00 ┆ 1      ┆ null │
│ 2024-01-01 00:02:00 ┆ 2      ┆ null │
│ 2024-01-01 00:03:00 ┆ 3      ┆ null │
│ 2024-01-01 00:04:00 ┆ 4      ┆ null │
│ …                   ┆ …      ┆ …    │
│ 2024-01-01 16:35:00 ┆ 995    ┆ null │
│ 2024-01-01 16:36:00 ┆ 996    ┆ null │
│ 2024-01-01 16:37:00 ┆ 997    ┆ null │
│ 2024-01-01 16:38:00 ┆ 998    ┆ null │
│ 2024-01-01 16:39:00 ┆ 999    ┆ null │
└─────────────────────┴────────┴──────┘


In [78]:
# df = result
time_window = timedelta(minutes=3)
def calculate_prob(time_delta: timedelta, sigma = 0.1) -> float:
    # gaussian distribution
    return np.exp(-time_delta ** 2 / (2 * sigma**2))
    
for row in df.filter(pl.col("inc").is_not_null()).iter_rows(named=True):
    timestamp = row["timestamp"]
    # Define the time window
    start_time = timestamp - time_window
    end_time = timestamp + time_window
    # Apply new value to neighbours within the time window
    df = df.with_columns(
            pl.when(
                (pl.col("timestamp") >= start_time) & 
                (pl.col("timestamp") <= end_time) & 
                pl.col("inc").is_null()
            ).then(row["inc"]).otherwise(pl.col("inc")).alias("inc")
        )




In [11]:
timestamp_col = 'ImageMetadata_time_stamp'
incident_type_col = "incident_type"
for row in df.filter(pl.col(incident_type_col).is_not_null()).iter_rows(
    named=True
):
    timestamp = row[timestamp_col]
    time_delta = timedelta(seconds=2)
    df = df.with_columns(
        pl.when(
            (pl.col(timestamp_col) >= timestamp - time_delta) &
            (pl.col(timestamp_col) <= timestamp + time_delta) &
             pl.col(incident_type_col).is_null()
        )
        .then(pl.lit(row[incident_type_col]))
        .otherwise(pl.col(incident_type_col))
        .alias(incident_type_col)
    )


In [40]:
df.with_columns(pl.when(pl.col(incident_type_col).is_not_null()).then(pl.lit(1)).otherwise(pl.lit(0)).alias("incident_probability"))

ImageMetadata_time_stamp,ImageMetadata_camera_name,ImageMetadata_frame_idx,Gnss_heading,Gnss_heading_error,Gnss_hp_loc_altitude,Gnss_hp_loc_latitude,Gnss_hp_loc_longitude,Gnss_position_covariance,Gnss_status,VehicleMotion_brake_pedal_normalized,VehicleMotion_gas_pedal_normalized,VehicleMotion_speed,VehicleMotion_steering_angle_normalized,VehicleMotion_gear,VehicleState_turn_signal,incident_type,incident_probability
datetime[μs],enum,i32,f64,f64,f64,f64,f64,list[f64],enum,f64,f64,f64,f64,enum,enum,enum,i32
2024-04-22 13:35:10.037599,"""cam_front_left""",0,256.44688,1.00789,47.052,53.844965,10.703684,"[63724.5, 63724.5, 268324.0]","""1""",0.0,0.066929,18.398406,0.005,"""3""","""0""",null,0
2024-04-22 13:35:10.062739,"""cam_front_left""",1,254.79102,1.02158,47.239,53.844964,10.703677,"[63724.5, 63724.5, 268324.0]","""1""",0.0,0.066929,18.355581,0.005081,"""3""","""0""",null,0
2024-04-22 13:35:10.096035,"""cam_front_left""",2,254.79102,1.02158,47.239,53.844964,10.703677,"[63724.5, 63724.5, 268324.0]","""1""",0.0,0.066929,18.320312,0.005434,"""3""","""0""",null,0
2024-04-22 13:35:10.129357,"""cam_front_left""",3,254.79102,1.02158,47.239,53.844964,10.703677,"[63724.5, 63724.5, 268324.0]","""1""",0.0,0.066929,18.307633,0.005833,"""3""","""0""",null,0
2024-04-22 13:35:10.162779,"""cam_front_left""",4,253.54586,1.01125,46.843,53.844963,10.70367,"[63724.5, 63724.5, 268324.0]","""1""",0.0,0.077761,18.347029,0.005833,"""3""","""0""",null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-04-22 14:56:40.208852,"""cam_front_left""",146676,151.87826,0.46759,52.0,53.848738,10.706358,"[39480.5, 39480.5, 147456.0]","""1""",0.0,0.110236,38.201077,0.01903,"""3""","""0""",null,0
2024-04-22 14:56:40.242211,"""cam_front_left""",146677,152.29572,0.47931,52.302,53.848729,10.706366,"[39762.0, 39762.0, 147456.0]","""1""",0.0,0.110236,38.193592,0.018125,"""3""","""0""",null,0
2024-04-22 14:56:40.275533,"""cam_front_left""",146678,152.29572,0.47931,52.302,53.848729,10.706366,"[39762.0, 39762.0, 147456.0]","""1""",0.0,0.110236,38.204758,0.017834,"""3""","""0""",null,0


In [39]:
df.filter(pl.col(incident_type_col).is_not_null())['incident_type']

incident_type
enum
"""Instructor Pedal"""
"""Instructor Pedal"""
"""Instructor Pedal"""
"""Instructor Pedal"""
"""Instructor Pedal"""
…
"""Instructor Pedal"""
"""Instructor Pedal"""
"""Instructor Pedal"""


In [106]:
df = df.with_columns(pl.col.incident_type.cast(str))

In [117]:
df.select(pl.col.incident_type).filter(pl.col.incident_type.is_not_null()).to_pandas().value_counts()

incident_type   
Instructor Pedal    11
Speed                3
Distance             1
Handling             1
Placement            1
Signs & Lights       1
Visual Check         1
Name: count, dtype: int64

In [8]:
_ = df.with_columns(
    pl.when(
        (pl.col(timestamp_col) >= timestamp - time_delta)
        & (pl.col(timestamp_col) <= timestamp + time_delta)
        & pl.col(incident_type_col).is_null()
    )
    .then(row[incident_type_col])
    .otherwise(pl.col(incident_type_col))
    .alias(incident_type_col)
    )

NameError: name 'time_delta' is not defined

In [5]:
df = pl.read_parquet('/home/nikita/yaak-datasets/tmp/df.parquet')

In [77]:
df.select('inc').drop_nulls().unique()

inc
i64
3
7
4
2
1
5
8
9
0
